# Model 2 — Category Presence + Item-to-Lot Assignment (CP-SAT)

This notebook implements the second model with:

- **category–fixture presence variable** `x_ik`
- **item option–lot assignment variable** `z`
- **display quantity variable** `w`

where:

- `i` = category (`URUN ALT GRUBU`)
- `p` = item option within category
- `k` = fixture (`A_ij`)
- `l` = lot within fixture

The model allows **multiple categories inside one fixture**, but uses `x_ik` to explicitly show whether category `i` is present in fixture `k`.

In [1]:
# ============================================================
# 1) IMPORTS
# ============================================================
import random
import numpy as np
import pandas as pd
import re
import unicodedata
from collections import defaultdict
from ortools.sat.python import cp_model

# GLOBAL
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [2]:
# ============================================================
# 2) FILE PATHS AND SETTINGS
# ============================================================
LOTS_FILE = "../../data/03_processed/lots_with_Aij.xlsx"
PRODUCT_FILE = "../../data/02_interim/saleability_scores_full_M9.xlsx"
OUTPUT_EXCEL = "../../data/03_processed/kontrol.xlsx"

USE_TIE_BREAK = True
MAX_TIME_SECONDS = 120
NUM_WORKERS = 16
LOG_SEARCH_PROGRESS = True

In [3]:
# ============================================================
# 3) LOAD DATA
# ============================================================
lots_df = pd.read_excel(LOTS_FILE).copy()
items_df = pd.read_excel(PRODUCT_FILE).copy()

print("Lots shape:", lots_df.shape)
print("Items shape:", items_df.shape)

Lots shape: (563, 11)
Items shape: (586, 14)


In [4]:
# ============================================================
# 4) REQUIRED COLUMNS
# ============================================================
required_lot_cols = [
    "lot_ID", "A_ij", "A_i", "A_j", "fixture_type", "r", "c", "mode", "L_type_score"
]

required_item_cols = [
    "ItemOption", "URUN ALT GRUBU", "LifeStyleGroup",
    "ETIKET", "discount", "saleability_score_full_M9_orig"
]

missing_lot = [c for c in required_lot_cols if c not in lots_df.columns]
missing_item = [c for c in required_item_cols if c not in items_df.columns]

if missing_lot:
    raise ValueError(f"Missing lot columns: {missing_lot}")

if missing_item:
    raise ValueError(f"Missing item columns: {missing_item}")

print("All required columns are present.")

All required columns are present.


In [5]:
# ============================================================
# 5) COLUMN DEFINITIONS
# ============================================================
ITEM_ID_COL = "ItemOption"
CATEGORY_COL = "URUN ALT GRUBU"
LIFESTYLE_COL = "LifeStyleGroup"

BASE_PRICE_COL = "ETIKET"
DISCOUNT_COL = "discount"
SALEABILITY_COL = "saleability_score_full_M9_orig"
LOT_SCORE_COL = "L_type_score"
FIXTURE_COL = "A_ij"

In [6]:
# ============================================================
# 6) HELPERS
# ============================================================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = x.replace("İ", "i").replace("I", "i").replace("ı", "i")
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    x = re.sub(r"\s+", " ", x).strip()
    return x

ACCESSORY_LIFESTYLE = normalize_text("aksesuar st")
SHOE_LIFESTYLE = normalize_text("ayakkabı st")


def clean_numeric_series(series):
    series = series.astype(str).str.replace(",", ".", regex=False)
    return pd.to_numeric(series, errors="coerce")


def is_hanger_fixture(fixture_type_norm):
    return fixture_type_norm in {
        "wall_hanger_small",
        "wall_hanger_large",
        "upright_small_3col",
    }


def is_table_or_4x4(fixture_type_norm):
    return fixture_type_norm.startswith("table_") or fixture_type_norm == "shelf_4x4"


def is_clothing_only_facing_fixture(fixture_type_norm):
    """One-row facing fixtures that can receive only clothing items."""
    return fixture_type_norm in {
        "single_facing_1x1",
        "double_facing_linear_1x10",
    }


def is_clothing_item(lifestyle_norm):
    """Clothing is defined as non-accessory and non-shoe lifestyle group."""
    return lifestyle_norm not in {ACCESSORY_LIFESTYLE, SHOE_LIFESTYLE}


In [7]:
# ============================================================
# 7) NUMERIC CLEANING
# ============================================================
items_df[BASE_PRICE_COL] = clean_numeric_series(items_df[BASE_PRICE_COL])
items_df[DISCOUNT_COL] = clean_numeric_series(items_df[DISCOUNT_COL])
items_df[SALEABILITY_COL] = clean_numeric_series(items_df[SALEABILITY_COL])
lots_df[LOT_SCORE_COL] = clean_numeric_series(lots_df[LOT_SCORE_COL])

lots_df["r"] = pd.to_numeric(lots_df["r"], errors="coerce")
lots_df["c"] = pd.to_numeric(lots_df["c"], errors="coerce")

items_df["discount_rate"] = items_df[DISCOUNT_COL].fillna(0).clip(lower=0, upper=1)
items_df["effective_price"] = items_df[BASE_PRICE_COL] * (1 - items_df["discount_rate"])

In [8]:
# ============================================================
# 8) DROP INVALID ROWS
# ============================================================
items_df = items_df.dropna(subset=[
    ITEM_ID_COL, CATEGORY_COL, LIFESTYLE_COL,
    BASE_PRICE_COL, DISCOUNT_COL, SALEABILITY_COL, "effective_price"
]).copy()

lots_df = lots_df.dropna(subset=[
    "lot_ID", FIXTURE_COL, "A_i", "A_j", "fixture_type", "r", "c", "mode", LOT_SCORE_COL
]).copy()

lots_df["r"] = lots_df["r"].astype(int)
lots_df["c"] = lots_df["c"].astype(int)

print("Valid items:", len(items_df))
print("Valid lots:", len(lots_df))

Valid items: 586
Valid lots: 563


In [9]:
# ============================================================
# 9) NORMALIZED TEXT COLUMNS
# ============================================================
items_df["category_norm"] = items_df[CATEGORY_COL].apply(normalize_text)
items_df["lifestyle_norm"] = items_df[LIFESTYLE_COL].apply(normalize_text)

lots_df["fixture_type_norm"] = lots_df["fixture_type"].apply(normalize_text)
lots_df["mode_norm"] = lots_df["mode"].apply(normalize_text)

In [10]:
# ============================================================
# 10) INDEXING
# ============================================================
items_df = items_df.reset_index(drop=True)
lots_df = lots_df.reset_index(drop=True)

items_df["item_idx"] = items_df.index
lots_df["lot_idx"] = lots_df.index

categories_df = (
    items_df[[CATEGORY_COL, "category_norm"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)
categories_df["cat_idx"] = categories_df.index

cat_norm_to_idx = dict(zip(categories_df["category_norm"], categories_df["cat_idx"]))
cat_idx_to_label = dict(zip(categories_df["cat_idx"], categories_df[CATEGORY_COL]))
cat_idx_to_norm = dict(zip(categories_df["cat_idx"], categories_df["category_norm"]))

items_df["cat_idx"] = items_df["category_norm"].map(cat_norm_to_idx)

fixtures_df = (
    lots_df[[FIXTURE_COL]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)
fixtures_df["fix_idx"] = fixtures_df.index

fix_value_to_idx = dict(zip(fixtures_df[FIXTURE_COL], fixtures_df["fix_idx"]))
fix_idx_to_value = dict(zip(fixtures_df["fix_idx"], fixtures_df[FIXTURE_COL]))

lots_df["fix_idx"] = lots_df[FIXTURE_COL].map(fix_value_to_idx)

CATS = categories_df["cat_idx"].tolist()
LOTS = lots_df["lot_idx"].tolist()
ITEMS = items_df["item_idx"].tolist()
FIXTURES = fixtures_df["fix_idx"].tolist()

print("|Categories| =", len(CATS))
print("|Fixtures|   =", len(FIXTURES))
print("|Lots|       =", len(LOTS))
print("|Items|      =", len(ITEMS))

|Categories| = 22
|Fixtures|   = 53
|Lots|       = 563
|Items|      = 586


In [11]:
# ============================================================
# 11) LOT DISPLAY CAPACITY c_l
# ============================================================
def get_lot_capacity(lot_row):
    fixture = lot_row["fixture_type_norm"]
    r = int(lot_row["r"])

    if is_table_or_4x4(fixture):
        return 3

    if is_hanger_fixture(fixture):
        if r == 1:   # bottom -> shoes
            return 1
        elif r == 2: # middle -> apparel
            return 2
        elif r == 3: # top -> accessories
            return 1

    return 1

lots_df["c_l"] = lots_df.apply(get_lot_capacity, axis=1)
print(lots_df["c_l"].value_counts(dropna=False))

c_l
2    291
1    182
3     90
Name: count, dtype: int64


In [12]:
# ============================================================
# 12) ELIGIBILITY RULES
# ============================================================
allowed_table_subgroups = {
    normalize_text("triko"),
    normalize_text("pantolon"),
    normalize_text("jean"),
    normalize_text("bluz"),
    normalize_text("gömlek"),
    normalize_text("gomlek"),
    normalize_text("tshirt"),
    normalize_text("tişört"),
    normalize_text("tisort"),
    normalize_text("sweatshirt"),
}


def eligible_item_for_lot(item_row, lot_row):
    fixture = lot_row["fixture_type_norm"]
    r = int(lot_row["r"])
    category = item_row["category_norm"]
    lifestyle = item_row["lifestyle_norm"]

    # Fixture-type rule: these two one-row facing fixtures are clothing-only.
    # This is separate from hanger row/mode logic and does not modify hanger rules.
    if is_clothing_only_facing_fixture(fixture):
        return is_clothing_item(lifestyle)

    if is_hanger_fixture(fixture):
        if r == 1:
            return lifestyle == SHOE_LIFESTYLE
        elif r == 2:
            return is_clothing_item(lifestyle)
        elif r == 3:
            return lifestyle == ACCESSORY_LIFESTYLE
        else:
            return False

    if is_table_or_4x4(fixture):
        if category == normalize_text("etek"):
            return False
        return category in allowed_table_subgroups

    return True


In [13]:
# ============================================================
# 13) BUILD FEASIBLE ITEM-LOT PAIRS
# z_(i,p,k,l) is represented in code as z[(item_idx, lot_idx)]
# ============================================================
OBJ_SCALE = 10000

feasible_item_lot_pairs = []
raw_unit_value_dict = {}
scaled_unit_value_dict = {}

for _, item in items_df.iterrows():
    item_idx = int(item["item_idx"])
    price_i = float(item["effective_price"])
    sale_i = float(item[SALEABILITY_COL])

    for _, lot in lots_df.iterrows():
        lot_idx = int(lot["lot_idx"])

        if eligible_item_for_lot(item, lot):
            loc_score = float(lot[LOT_SCORE_COL])
            unit_value = price_i * sale_i * loc_score

            feasible_item_lot_pairs.append((item_idx, lot_idx))
            raw_unit_value_dict[(item_idx, lot_idx)] = unit_value
            scaled_unit_value_dict[(item_idx, lot_idx)] = int(round(unit_value * OBJ_SCALE))

clothing_only_lot_idxs = set(
    lots_df.loc[
        lots_df["fixture_type_norm"].apply(is_clothing_only_facing_fixture),
        "lot_idx"
    ].astype(int)
)
non_clothing_item_idxs = set(
    items_df.loc[
        ~items_df["lifestyle_norm"].apply(is_clothing_item),
        "item_idx"
    ].astype(int)
)
invalid_clothing_only_pairs = [
    (item_idx, lot_idx)
    for item_idx, lot_idx in feasible_item_lot_pairs
    if lot_idx in clothing_only_lot_idxs and item_idx in non_clothing_item_idxs
]

if invalid_clothing_only_pairs:
    raise ValueError(
        "Clothing-only facing fixture rule failed: "
        f"{len(invalid_clothing_only_pairs)} invalid item-lot pairs were created."
    )

print("Feasible item-lot pairs:", len(feasible_item_lot_pairs))
print("Clothing-only facing fixture rule validated for single_facing_1x1 and double_facing_linear_1x10.")


Feasible item-lot pairs: 164194
Clothing-only facing fixture rule validated for single_facing_1x1 and double_facing_linear_1x10.


In [14]:
# ============================================================
# 14) BUILD FEASIBLE CATEGORY-FIXTURE PAIRS
# ============================================================
cat_fix_possible = defaultdict(int)

for item_idx, lot_idx in feasible_item_lot_pairs:
    item_row = items_df.loc[items_df["item_idx"] == item_idx].iloc[0]
    lot_row = lots_df.loc[lots_df["lot_idx"] == lot_idx].iloc[0]

    cat_idx = int(item_row["cat_idx"])
    fix_idx = int(lot_row["fix_idx"])
    cat_fix_possible[(cat_idx, fix_idx)] = 1

feasible_cat_fix_pairs = sorted(cat_fix_possible.keys())
print("Feasible category-fixture pairs:", len(feasible_cat_fix_pairs))

Feasible category-fixture pairs: 970


In [15]:
# ============================================================
# 15) PREPARE GROUPINGS
# ============================================================
pairs_by_lot = defaultdict(list)
pairs_by_item = defaultdict(list)
pairs_by_cat_fix = defaultdict(list)

items_by_category = items_df.groupby("cat_idx")["item_idx"].apply(list).to_dict()

for item_idx, lot_idx in feasible_item_lot_pairs:
    item_row = items_df.loc[items_df["item_idx"] == item_idx].iloc[0]
    lot_row = lots_df.loc[lots_df["lot_idx"] == lot_idx].iloc[0]

    cat_idx = int(item_row["cat_idx"])
    fix_idx = int(lot_row["fix_idx"])

    pairs_by_lot[lot_idx].append((item_idx, lot_idx))
    pairs_by_item[item_idx].append((item_idx, lot_idx))
    pairs_by_cat_fix[(cat_idx, fix_idx)].append((item_idx, lot_idx))

infeasible_lots = [lot for lot in LOTS if len(pairs_by_lot[lot]) == 0]
if infeasible_lots:
    raise ValueError(f"Some lots have no feasible items: {infeasible_lots[:20]}")

infeasible_categories = [cat for cat in CATS if len([k for (i, k) in feasible_cat_fix_pairs if i == cat]) == 0]
if infeasible_categories:
    labels = [cat_idx_to_label[c] for c in infeasible_categories]
    raise ValueError(f"Some categories cannot appear in any fixture: {labels}")

In [16]:
# ============================================================
# 16) MODEL
# ============================================================
model = cp_model.CpModel()

x = {}
for cat_idx, fix_idx in feasible_cat_fix_pairs:
    x[(cat_idx, fix_idx)] = model.NewBoolVar(f"x_cat{cat_idx}_fix{fix_idx}")

z = {}
w = {}

lot_capacity = lots_df.set_index("lot_idx")["c_l"].to_dict()

for item_idx, lot_idx in feasible_item_lot_pairs:
    z[(item_idx, lot_idx)] = model.NewBoolVar(f"z_item{item_idx}_lot{lot_idx}")
    w[(item_idx, lot_idx)] = model.NewIntVar(0, int(lot_capacity[lot_idx]), f"w_item{item_idx}_lot{lot_idx}")

In [17]:
# ============================================================
# 17) OBJECTIVE
# ============================================================
main_objective = sum(
    scaled_unit_value_dict[(item_idx, lot_idx)] * w[(item_idx, lot_idx)]
    for (item_idx, lot_idx) in feasible_item_lot_pairs
)

if USE_TIE_BREAK:
    model.Maximize(
        main_objective * 1000
        - sum((item_idx + lot_idx) * z[(item_idx, lot_idx)] for (item_idx, lot_idx) in feasible_item_lot_pairs)
    )
else:
    model.Maximize(main_objective)

In [18]:
# ============================================================
# 18) CONSTRAINTS
# ============================================================
# every fixture must contain at least one category
for fix_idx in FIXTURES:
    active_categories = [x[(cat_idx, fix_idx)] for (cat_idx, k) in feasible_cat_fix_pairs if k == fix_idx]
    if not active_categories:
        raise ValueError(f"Fixture {fix_idx_to_value[fix_idx]} has no feasible category.")
    model.Add(sum(active_categories) >= 1)

# every category must appear in at least one fixture
for cat_idx in CATS:
    possible_fixtures = [x[(cat_idx, fix_idx)] for (i, fix_idx) in feasible_cat_fix_pairs if i == cat_idx]
    if not possible_fixtures:
        raise ValueError(f"Category {cat_idx_to_label[cat_idx]} has no feasible fixture.")
    model.Add(sum(possible_fixtures) >= 1)

# every lot must be filled with exactly one item option
for lot_idx in LOTS:
    model.Add(sum(z[(item_idx, lot_idx)] for (item_idx, lot_idx) in pairs_by_lot[lot_idx]) == 1)

# each item option can be assigned at most once
for item_idx in ITEMS:
    if pairs_by_item[item_idx]:
        model.Add(sum(z[(item_idx, lot_idx)] for (item_idx, lot_idx) in pairs_by_item[item_idx]) <= 1)

# quantity linking
for item_idx, lot_idx in feasible_item_lot_pairs:
    model.Add(w[(item_idx, lot_idx)] == int(lot_capacity[lot_idx]) * z[(item_idx, lot_idx)])

# z can be 1 only if the category is present in that fixture
for item_idx, lot_idx in feasible_item_lot_pairs:
    item_row = items_df.loc[items_df["item_idx"] == item_idx].iloc[0]
    lot_row = lots_df.loc[lots_df["lot_idx"] == lot_idx].iloc[0]

    cat_idx = int(item_row["cat_idx"])
    fix_idx = int(lot_row["fix_idx"])
    model.Add(z[(item_idx, lot_idx)] <= x[(cat_idx, fix_idx)])

# strong reverse linking: if x_ik = 1, there must be at least one assigned item of that category in that fixture
for cat_idx, fix_idx in feasible_cat_fix_pairs:
    related_z = pairs_by_cat_fix[(cat_idx, fix_idx)]
    model.Add(x[(cat_idx, fix_idx)] <= sum(z[(item_idx, lot_idx)] for (item_idx, lot_idx) in related_z))

# same category can appear at most 3 times in one fixture
for cat_idx, fix_idx in feasible_cat_fix_pairs:
    related_z = pairs_by_cat_fix[(cat_idx, fix_idx)]
    if related_z:
        model.Add(sum(z[(item_idx, lot_idx)] for (item_idx, lot_idx) in related_z) <= 3)

# no 3 consecutive lots in same row within same fixture can have the same category
for fix_idx, fixture_group in lots_df.groupby("fix_idx"):
    for row_r, row_group in fixture_group.groupby("r"):
        row_group = row_group.sort_values("c").copy()

        col_to_lot = dict(zip(row_group["c"], row_group["lot_idx"]))
        cols_sorted = sorted(col_to_lot.keys())

        for idx in range(len(cols_sorted) - 2):
            c1, c2, c3 = cols_sorted[idx], cols_sorted[idx + 1], cols_sorted[idx + 2]

            if not (c2 == c1 + 1 and c3 == c2 + 1):
                continue

            lot1 = col_to_lot[c1]
            lot2 = col_to_lot[c2]
            lot3 = col_to_lot[c3]

            for cat_idx in CATS:
                triplet_terms = []

                for item_idx in items_by_category.get(cat_idx, []):
                    if (item_idx, lot1) in z:
                        triplet_terms.append(z[(item_idx, lot1)])
                    if (item_idx, lot2) in z:
                        triplet_terms.append(z[(item_idx, lot2)])
                    if (item_idx, lot3) in z:
                        triplet_terms.append(z[(item_idx, lot3)])

                if triplet_terms:
                    model.Add(sum(triplet_terms) <= 2)

In [19]:
# ============================================================
# PRE-SOLVE DIAGNOSTIC
# ============================================================

print("Total lots:", len(LOTS))
print("Total items:", len(ITEMS))
print("Feasible item-lot pairs:", len(feasible_item_lot_pairs))

# Lot başına kaç uygun item var?
lot_candidate_counts = pd.DataFrame([
    {
        "lot_idx": lot_idx,
        "candidate_count": len(pairs_by_lot[lot_idx]),
        "fixture_type": lots_df.loc[lots_df["lot_idx"] == lot_idx, "fixture_type"].iloc[0],
        "fixture_A_ij": lots_df.loc[lots_df["lot_idx"] == lot_idx, FIXTURE_COL].iloc[0],
        "r": lots_df.loc[lots_df["lot_idx"] == lot_idx, "r"].iloc[0],
        "c": lots_df.loc[lots_df["lot_idx"] == lot_idx, "c"].iloc[0],
        "mode": lots_df.loc[lots_df["lot_idx"] == lot_idx, "mode"].iloc[0],
    }
    for lot_idx in LOTS
]).sort_values("candidate_count")

display(lot_candidate_counts.head(30))

# Clothing-only facing lotları özel kontrol
facing_lots = lots_df[
    lots_df["fixture_type_norm"].apply(is_clothing_only_facing_fixture)
].copy()

print("Clothing-only facing lots:", len(facing_lots))

facing_lot_candidate_counts = lot_candidate_counts[
    lot_candidate_counts["fixture_type"].isin(["single_facing_1x1", "double_facing_linear_1x10"])
].copy()

display(facing_lot_candidate_counts.sort_values("candidate_count"))

# Uygun clothing item sayısı
clothing_items = items_df[items_df["lifestyle_norm"].apply(is_clothing_item)].copy()
print("Total clothing item options:", len(clothing_items))

# Facing fixture bazında kaç lot var?
facing_fixture_lot_counts = (
    facing_lots.groupby([FIXTURE_COL, "fixture_type"])
    .size()
    .rename("lot_count")
    .reset_index()
    .sort_values("lot_count", ascending=False)
)

display(facing_fixture_lot_counts)

Total lots: 563
Total items: 586
Feasible item-lot pairs: 164194


,lot_idx,candidate_count,fixture_type,fixture_A_ij,r,c,mode
562,562,87,wall_hanger_small,"A(40,4)",1,1,normal
561,561,87,wall_hanger_small,"A(39,12)",1,1,normal
560,560,87,wall_hanger_large,"A(39,1)",1,1,normal
559,559,87,wall_hanger_small,"A(37,12)",1,1,normal
558,558,87,wall_hanger_large,"A(37,1)",1,1,normal
557,557,87,upright_small_3col,"A(35,2)",1,3,normal
556,556,87,upright_small_3col,"A(35,2)",1,1,normal
555,555,87,upright_small_3col,"A(34,8)",1,3,normal
494,494,87,wall_hanger_small,"A(11,12)",1,2,normal
495,495,87,wall_hanger_small,"A(12,11)",1,2,normal


Clothing-only facing lots: 24


,lot_idx,candidate_count,fixture_type,fixture_A_ij,r,c,mode
235,235,405,double_facing_linear_1x10,"A(38,6)",1,3,normal
236,236,405,double_facing_linear_1x10,"A(38,6)",1,8,normal
237,237,405,double_facing_linear_1x10,"A(38,7)",1,3,normal
238,238,405,double_facing_linear_1x10,"A(38,7)",1,8,normal
293,293,405,double_facing_linear_1x10,"A(38,6)",1,2,normal
294,294,405,double_facing_linear_1x10,"A(38,6)",1,9,normal
295,295,405,double_facing_linear_1x10,"A(38,7)",1,2,normal
296,296,405,double_facing_linear_1x10,"A(38,7)",1,9,normal
195,195,405,double_facing_linear_1x10,"A(38,7)",1,4,normal
196,196,405,double_facing_linear_1x10,"A(38,7)",1,7,normal


Total clothing item options: 405


,A_ij,fixture_type,lot_count
3,"A(38,7)",double_facing_linear_1x10,10
2,"A(38,6)",double_facing_linear_1x10,10
1,"A(26,9)",single_facing_1x1,1
0,"A(21,9)",single_facing_1x1,1
4,"A(6,12)",single_facing_1x1,1
5,"A(7,12)",single_facing_1x1,1


In [20]:
# ============================================================
# 19) SOLVE
# ============================================================
solver = cp_model.CpSolver()

solver.parameters.random_seed = SEED
solver.parameters.max_time_in_seconds = MAX_TIME_SECONDS
solver.parameters.num_search_workers = NUM_WORKERS
solver.parameters.log_search_progress = LOG_SEARCH_PROGRESS

solver.parameters.randomize_search = False
solver.parameters.search_branching = cp_model.FIXED_SEARCH

status = solver.Solve(model)

status_map = {
    cp_model.OPTIMAL: "OPTIMAL",
    cp_model.FEASIBLE: "FEASIBLE",
    cp_model.INFEASIBLE: "INFEASIBLE",
    cp_model.MODEL_INVALID: "MODEL_INVALID",
    cp_model.UNKNOWN: "UNKNOWN",
}
print("Solver status:", status_map.get(status, status))

if status not in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
    raise RuntimeError("No feasible solution found.")

Solver status: FEASIBLE


In [21]:
# ============================================================
# 20) EXTRACT x_ik PRESENCE RESULTS
# ============================================================
x_rows = []

for cat_idx, fix_idx in feasible_cat_fix_pairs:
    x_val = solver.Value(x[(cat_idx, fix_idx)])
    if x_val == 1:
        x_rows.append({
            "cat_idx": cat_idx,
            "category": cat_idx_to_label[cat_idx],
            "fix_idx": fix_idx,
            "fixture_A_ij": fix_idx_to_value[fix_idx],
            "x_ik": x_val,
            "var_name_x": f"x_cat{cat_idx}_fix{fix_idx}",
        })

x_presence_df = pd.DataFrame(x_rows).sort_values(["fixture_A_ij", "category"]).reset_index(drop=True)
x_presence_df.head()

,cat_idx,category,fix_idx,fixture_A_ij,x_ik,var_name_x
0,21,CANTA,10,"A(1,11)",1,x_cat21_fix10
1,9,CEKET,10,"A(1,11)",1,x_cat9_fix10
2,3,GOMLEK,10,"A(1,11)",1,x_cat3_fix10
3,10,KABAN,10,"A(1,11)",1,x_cat10_fix10
4,5,MONT,10,"A(1,11)",1,x_cat5_fix10


In [22]:
# ============================================================
# 21) EXTRACT z AND w RESULTS
# ============================================================
selected_rows = []
all_decision_rows = []

for item_idx, lot_idx in feasible_item_lot_pairs:
    z_val = solver.Value(z[(item_idx, lot_idx)])
    w_val = solver.Value(w[(item_idx, lot_idx)])

    item_row = items_df.loc[items_df["item_idx"] == item_idx].iloc[0]
    lot_row = lots_df.loc[lots_df["lot_idx"] == lot_idx].iloc[0]

    cat_idx = int(item_row["cat_idx"])
    fix_idx = int(lot_row["fix_idx"])

    unit_value = raw_unit_value_dict[(item_idx, lot_idx)]
    total_contribution = unit_value * w_val

    all_decision_rows.append({
        "cat_idx": cat_idx,
        "category": item_row[CATEGORY_COL],
        "item_idx": item_idx,
        "ItemOption": item_row[ITEM_ID_COL],
        "fix_idx": fix_idx,
        "fixture_A_ij": lot_row[FIXTURE_COL],
        "lot_idx": lot_idx,
        "lot_ID": lot_row["lot_ID"],
        "r": int(lot_row["r"]),
        "c": int(lot_row["c"]),
        "mode": lot_row["mode"],
        "c_l": int(lot_capacity[lot_idx]),
        "z_value": z_val,
        "w_value": w_val,
        "var_name_z": f"z_item{item_idx}_lot{lot_idx}",
        "var_name_w": f"w_item{item_idx}_lot{lot_idx}",
    })

    if z_val == 1:
        selected_rows.append({
            "cat_idx": cat_idx,
            "category": item_row[CATEGORY_COL],
            "item_idx": item_idx,
            "ItemOption": item_row[ITEM_ID_COL],
            "LifeStyleGroup": item_row[LIFESTYLE_COL],
            "base_price": float(item_row[BASE_PRICE_COL]),
            "discount_rate": float(item_row["discount_rate"]),
            "effective_price": float(item_row["effective_price"]),
            "saleability_score": float(item_row[SALEABILITY_COL]),
            "fix_idx": fix_idx,
            "fixture_A_ij": lot_row[FIXTURE_COL],
            "A_i": lot_row["A_i"],
            "A_j": lot_row["A_j"],
            "lot_idx": lot_idx,
            "lot_ID": lot_row["lot_ID"],
            "fixture_type": lot_row["fixture_type"],
            "r": int(lot_row["r"]),
            "c": int(lot_row["c"]),
            "mode": lot_row["mode"],
            "L_type_score": float(lot_row[LOT_SCORE_COL]),
            "c_l": int(lot_capacity[lot_idx]),
            "x_ik": solver.Value(x[(cat_idx, fix_idx)]),
            "z_value": z_val,
            "w_value": w_val,
            "unit_objective_value": unit_value,
            "objective_contribution_raw": total_contribution,
            "objective_contribution_scaled": scaled_unit_value_dict[(item_idx, lot_idx)] * w_val,
            "var_name_x": f"x_cat{cat_idx}_fix{fix_idx}",
            "var_name_z": f"z_item{item_idx}_lot{lot_idx}",
            "var_name_w": f"w_item{item_idx}_lot{lot_idx}",
        })

selected_df = pd.DataFrame(selected_rows)
all_decision_df = pd.DataFrame(all_decision_rows)

print("Selected assignments:", len(selected_df))
selected_df.head()

Selected assignments: 563


,cat_idx,category,item_idx,ItemOption,LifeStyleGroup,base_price,discount_rate,effective_price,saleability_score,fix_idx,...,c_l,x_ik,z_value,w_value,unit_objective_value,objective_contribution_raw,objective_contribution_scaled,var_name_x,var_name_z,var_name_w
0,0,TISORT,0,nd434,Mono,699.0,0.18,573.18,1.000000,6,...,3,1,1,3,619.034400,1857.103200,18571032,x_cat0_fix6,z_item0_lot8,w_item0_lot8
1,0,TISORT,1,nd448,Business,999.0,0.14,859.14,0.790935,39,...,2,1,1,2,645.547460,1291.094920,12910950,x_cat0_fix39,z_item1_lot87,w_item1_lot87
2,0,TISORT,2,nd468,Mono,699.0,0.34,461.34,0.730703,37,...,2,1,1,2,316.033691,632.067381,6320674,x_cat0_fix37,z_item2_lot116,w_item2_lot116
3,1,PANTOLON,3,nd266,Essential,1799.0,0.08,1655.08,0.645277,12,...,2,1,1,2,1001.236502,2002.473005,20024730,x_cat1_fix12,z_item3_lot91,w_item3_lot91
4,2,TRIKO,4,nd85,Essential,1699.0,0.15,1444.15,0.638807,39,...,2,1,1,2,830.279797,1660.559595,16605596,x_cat2_fix39,z_item4_lot191,w_item4_lot191


In [23]:
# ============================================================
# 22) READABLE OUTPUT TABLES
# ============================================================
assignment_table = selected_df[[
    "category",
    "ItemOption",
    "LifeStyleGroup",
    "base_price",
    "discount_rate",
    "effective_price",
    "saleability_score",
    "fixture_A_ij",
    "A_i",
    "A_j",
    "lot_ID",
    "fixture_type",
    "r",
    "c",
    "mode",
    "L_type_score",
    "c_l",
    "x_ik",
    "z_value",
    "w_value",
    "unit_objective_value",
    "objective_contribution_raw",
    "var_name_x",
    "var_name_z",
    "var_name_w",
]].sort_values(["fixture_A_ij", "r", "c"]).reset_index(drop=True)

contribution_report = selected_df[[
    "category",
    "ItemOption",
    "LifeStyleGroup",
    "base_price",
    "discount_rate",
    "effective_price",
    "saleability_score",
    "fixture_A_ij",
    "lot_ID",
    "fixture_type",
    "r",
    "c",
    "mode",
    "L_type_score",
    "c_l",
    "w_value",
    "unit_objective_value",
    "objective_contribution_raw",
]].sort_values("objective_contribution_raw", ascending=False).reset_index(drop=True)

z_selected_only = all_decision_df[all_decision_df["z_value"] == 1].copy()
assignment_table.head()

,category,ItemOption,LifeStyleGroup,base_price,discount_rate,effective_price,saleability_score,fixture_A_ij,A_i,A_j,...,L_type_score,c_l,x_ik,z_value,w_value,unit_objective_value,objective_contribution_raw,var_name_x,var_name_z,var_name_w
0,ÇİZME,nd389,Ayakkabı ST,3999.0,0.63,1479.63,0.043926,"A(1,11)",1,11,...,0.4680,1,1,1,1,30.417443,30.417443,x_cat17_fix10,z_item456_lot521,w_item456_lot521
1,TOPUKLU AYAKKABI,nd519,Ayakkabı ST,2599.0,0.00,2599.00,0.038356,"A(1,11)",1,11,...,0.5178,1,1,1,1,51.618699,51.618699,x_cat20_fix10,z_item469_lot485,w_item469_lot485
2,KABAN,nd43,Tema,15999.0,0.62,6079.62,0.054820,"A(1,11)",1,11,...,0.7500,2,1,1,2,249.963109,499.926219,x_cat10_fix10,z_item428_lot328,w_item428_lot328
3,PANTOLON,nd268,Mono,3599.0,0.47,1907.47,0.325605,"A(1,11)",1,11,...,0.8333,2,1,1,2,517.547925,1035.095850,x_cat1_fix10,z_item93_lot247,w_item93_lot247
4,GOMLEK,nd234,Mono,5599.0,0.09,5095.09,0.308287,"A(1,11)",1,11,...,0.9167,2,1,1,2,1439.904901,2879.809802,x_cat3_fix10,z_item105_lot127,w_item105_lot127


In [24]:
# ============================================================
# 23) DIAGNOSTICS
# ============================================================
lot_fill_check = (
    selected_df.groupby("lot_idx").size().rename("assigned_count").reset_index()
)

item_use_check = (
    selected_df.groupby("item_idx").size().rename("used_count").reset_index()
)

category_presence_check = (
    x_presence_df.groupby("category").size().rename("fixture_count").reset_index()
)

category_count_in_fixture = (
    selected_df.groupby(["fixture_A_ij", "category"])
    .size()
    .rename("count_in_fixture")
    .reset_index()
    .sort_values(["fixture_A_ij", "count_in_fixture"], ascending=[True, False])
)

print("=" * 60)
print("OBJECTIVE SUMMARY")
print("=" * 60)
print("Scaled objective value:", solver.ObjectiveValue())
print("Approximate raw objective value:", selected_df["objective_contribution_raw"].sum())
print("Total displayed quantity:", selected_df["w_value"].sum())

print("\n" + "=" * 60)
print("LOT FILL CHECK")
print("=" * 60)
print(lot_fill_check["assigned_count"].value_counts(dropna=False))

print("\n" + "=" * 60)
print("ITEM USE CHECK")
print("=" * 60)
print(item_use_check["used_count"].value_counts(dropna=False))

print("\n" + "=" * 60)
print("CATEGORY PRESENCE CHECK")
print("=" * 60)
print(category_presence_check.to_string(index=False))

OBJECTIVE SUMMARY
Scaled objective value: 6837914672322.0
Approximate raw objective value: 683791.4964435505
Total displayed quantity: 1034

LOT FILL CHECK
assigned_count
1    563
Name: count, dtype: int64

ITEM USE CHECK
used_count
1    563
Name: count, dtype: int64

CATEGORY PRESENCE CHECK
        category  fixture_count
          BLAZER              5
            BLUZ              9
             BOT              5
           CANTA             35
           CEKET             20
          CLUTCH             11
          ELBISE             16
            ETEK             17
          GOMLEK             26
   JEAN PANTOLON             16
           KABAN             13
            MONT             23
        PANTOLON             40
         PARDESU              4
         SNEAKER             11
      SWEATSHIRT             15
          TISORT             27
TOPUKLU AYAKKABI             34
           TRIKO             40
           TULUM              2
           YELEK              3
   

In [25]:
# ============================================================
# 24) EXPORT TO EXCEL
# ============================================================
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    x_presence_df.to_excel(writer, sheet_name="x_ik_presence", index=False)
    assignment_table.to_excel(writer, sheet_name="assignment_table", index=False)
    contribution_report.to_excel(writer, sheet_name="contribution_report", index=False)
    z_selected_only.to_excel(writer, sheet_name="selected_z_only", index=False)
    all_decision_df.to_excel(writer, sheet_name="all_z_long_format", index=False)
    lot_fill_check.to_excel(writer, sheet_name="lot_fill_check", index=False)
    item_use_check.to_excel(writer, sheet_name="item_use_check", index=False)
    category_presence_check.to_excel(writer, sheet_name="category_presence_check", index=False)
    category_count_in_fixture.to_excel(writer, sheet_name="category_count_fixture", index=False)

print(f"Results exported to: {OUTPUT_EXCEL}")

Results exported to: kontrol.xlsx


## Notes

- This model **does not force one category per fixture**.
- `x_ik` only indicates that category `i` is **present somewhere** in fixture `k`.
- The actual item-to-lot assignment is still done by `z`.
- The quantity logic is:
  - table / 4x4 = 3
  - hanger apparel row = 2
  - accessory = 1
  - shoes = 1

## Export assignment.json (for Step 4b: the store layout visualization)

Builds the nested fixture→lots JSON the `viz/store-planogram` React app fetches at runtime, by combining:
- `selected_df` / `assignment_table` from this notebook (the solved assignment)
- `items_df` for `ColorGroup` (not carried into `assignment_table` above)
- `../../data/02_interim/fixture_grid.json` from Step 2 for fixture geometry (`max_r`, `max_c`, `fix_idx`) and the store grid

Requires Step 2's export cell to have been run first.

In [ ]:
import json
from pathlib import Path

with open("../../data/02_interim/fixture_grid.json") as f:
    fixture_grid = json.load(f)

fx_by_aij = {fx["A_ij"]: fx for fx in fixture_grid["fixtures"]}

# Bring ColorGroup back in (present on items_df, dropped from assignment_table)
export_df = assignment_table.merge(
    items_df[["ItemOption", "ColorGroup"]], on="ItemOption", how="left"
)

lot_cols = [
    "lot_ID", "r", "c", "mode", "ItemOption", "category", "ColorGroup",
    "saleability_score", "L_type_score", "LifeStyleGroup",
    "base_price", "discount_rate", "effective_price",
]

fixtures_payload = []
for a_ij, group in export_df.groupby("fixture_A_ij"):
    fx_meta = fx_by_aij.get(a_ij)
    if fx_meta is None:
        raise KeyError(
            f"{a_ij} is in the assignment but missing from fixture_grid.json "
            "-- rerun the Step 2 export cell."
        )
    fixtures_payload.append({
        "A_ij": a_ij,
        "A_i": fx_meta["A_i"],
        "A_j": fx_meta["A_j"],
        "fixture_type": fx_meta["fixture_type"],
        "fix_idx": fx_meta["fix_idx"],
        "max_r": fx_meta["max_r"],
        "max_c": fx_meta["max_c"],
        "lots": group[lot_cols].sort_values(["r", "c"]).to_dict(orient="records"),
    })

assignment_payload = {
    "fixtures": fixtures_payload,
    "grid": fixture_grid["grid"],
    "N_I": fixture_grid["N_I"],
    "N_J": fixture_grid["N_J"],
}

out_paths = [
    Path("../../data/03_processed/assignment.json"),
    Path("../../viz/store-planogram/public/data/assignment.json"),
]
for out_path in out_paths:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(assignment_payload, f)

total_lots = sum(len(fx["lots"]) for fx in fixtures_payload)
print(f"Exported {len(fixtures_payload)} fixtures, {total_lots} lots to:")
for p in out_paths:
    print(" -", p)